<a href="https://colab.research.google.com/github/yvokafor-rgb/document-intelligent-system/blob/main/Advanced_Full_RAG_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

# Remove the broken/mixed Pillow installation
!pip uninstall -y pillow PIL

# Install compatible packages cleanly
!pip install -q --no-cache-dir --force-reinstall "pillow==11.3.0"

!pip install -q --no-cache-dir -U \
    gradio \
    pymupdf \
    pytesseract \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    sentencepiece

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 161.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 229.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 169.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 124.6 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import math
import tempfile
import traceback
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional

import fitz
import faiss
import torch
import numpy as np
import gradio as gr
import pytesseract

from PIL import Image

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [3]:
print("Gradio version:", gr.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Answers may take longer.")

Gradio version: 6.20.0
PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU. Answers may take longer.


In [4]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Good balance between size and answer quality
LLM_MODEL_NAME = "google/flan-t5-base"

CHUNK_SIZE = 180
CHUNK_OVERLAP = 40
DEFAULT_TOP_K = 2

# A digital page with less text than this may require OCR
MIN_DIGITAL_TEXT_LENGTH = 80

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Language model:", LLM_MODEL_NAME)
print("Device:", DEVICE)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Language model: google/flan-t5-base
Device: cpu


In [5]:
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded successfully.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [6]:
print("Loading open-source language model...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

if torch.cuda.is_available():
    llm_model = AutoModelForSeq2SeqLM.from_pretrained(
        LLM_MODEL_NAME,
        torch_dtype=torch.float16
    ).to(DEVICE)
else:
    llm_model = AutoModelForSeq2SeqLM.from_pretrained(
        LLM_MODEL_NAME
    ).to(DEVICE)

llm_model.eval()

print("Language model loaded successfully.")

Loading open-source language model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Language model loaded successfully.


In [7]:
def clean_text(text: str) -> str:
    """
    Clean text extracted from a PDF or OCR engine.
    """

    if not text:
        return ""

    # Remove null characters
    text = text.replace("\x00", " ")

    # Join words separated by a line-break hyphen
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Replace multiple line breaks with one space
    text = re.sub(r"\n+", " ", text)

    # Replace repeated spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    return text.strip()

In [8]:
def ocr_pdf_page(
    page,
    zoom: float = 2.0
) -> str:
    """
    Convert a PDF page into an image and extract its text with Tesseract.
    """

    matrix = fitz.Matrix(zoom, zoom)

    pixmap = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )

    image = Image.frombytes(
        "RGB",
        [pixmap.width, pixmap.height],
        pixmap.samples
    )

    ocr_text = pytesseract.image_to_string(
        image,
        config="--oem 3 --psm 6"
    )

    return clean_text(ocr_text)

In [9]:
def extract_pdf_pages(
    pdf_path: str,
    progress=None
) -> List[Dict]:
    """
    Extract every readable page from a PDF.

    Digital pages use PyMuPDF text extraction.
    Scanned pages use Tesseract OCR.
    """

    extracted_pages = []

    try:
        document = fitz.open(pdf_path)
    except Exception as error:
        raise ValueError(
            f"Could not open {os.path.basename(pdf_path)}: {error}"
        )

    total_pages = len(document)
    filename = os.path.basename(pdf_path)

    for page_index, page in enumerate(document):

        if progress is not None:
            progress(
                (page_index + 1) / max(total_pages, 1),
                desc=(
                    f"Reading {filename}: "
                    f"page {page_index + 1}/{total_pages}"
                )
            )

        # Try regular extraction first
        digital_text = clean_text(page.get_text("text"))

        if len(digital_text) >= MIN_DIGITAL_TEXT_LENGTH:
            page_text = digital_text
            extraction_method = "digital"
        else:
            # Use OCR for pages with little usable digital text
            page_text = ocr_pdf_page(page)
            extraction_method = "ocr"

            # Fall back to digital text if OCR unexpectedly returns less
            if len(page_text) < len(digital_text):
                page_text = digital_text
                extraction_method = "digital"

        if page_text.strip():
            extracted_pages.append({
                "filename": filename,
                "source_id": Path(filename).stem,
                "page_number": page_index + 1,
                "total_pages": total_pages,
                "text": page_text,
                "extraction_method": extraction_method
            })

    document.close()

    return extracted_pages

In [10]:
DOCUMENT_TYPE_KEYWORDS = {
    "loan_estimate": [
        "loan estimate",
        "estimated total payment",
        "projected payments",
        "closing cost details",
        "loan terms"
    ],

    "closing_disclosure": [
        "closing disclosure",
        "cash to close",
        "loan calculations",
        "summaries of transactions"
    ],

    "fees_worksheet": [
        "fees worksheet",
        "fee worksheet",
        "origination charges",
        "other charges",
        "estimated funds to close"
    ],

    "mortgage_statement": [
        "mortgage statement",
        "amount due",
        "payment due date",
        "principal balance",
        "escrow balance"
    ],

    "promissory_note": [
        "promissory note",
        "borrower's promise to pay",
        "interest rate",
        "payment schedule"
    ],

    "deed_of_trust": [
        "deed of trust",
        "security instrument",
        "trustee",
        "property address"
    ],

    "credit_report": [
        "credit report",
        "credit score",
        "tradeline",
        "payment history"
    ],

    "appraisal": [
        "appraisal report",
        "market value",
        "comparable sales",
        "subject property"
    ],

    "bank_statement": [
        "bank statement",
        "account summary",
        "beginning balance",
        "ending balance",
        "deposits and withdrawals"
    ],

    "income_document": [
        "w-2",
        "w2 wage",
        "pay stub",
        "gross pay",
        "year to date earnings",
        "tax return"
    ]
}

In [11]:
def classify_document_type(
    text: str,
    filename: str = ""
) -> str:
    """
    Predict a mortgage-document type using filename and text keywords.
    """

    searchable_text = (
        f"{filename} {text[:5000]}"
    ).lower()

    scores = {}

    for doc_type, keywords in DOCUMENT_TYPE_KEYWORDS.items():
        score = 0

        for keyword in keywords:
            if keyword in searchable_text:
                score += 1

        scores[doc_type] = score

    best_type = max(scores, key=scores.get)

    if scores[best_type] == 0:
        return "other_mortgage_document"

    return best_type

In [12]:
def split_text_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP
) -> List[str]:
    """
    Split text into overlapping word chunks.
    """

    words = text.split()

    if not words:
        return []

    if overlap >= chunk_size:
        raise ValueError(
            "Chunk overlap must be smaller than chunk size."
        )

    chunks = []
    step_size = chunk_size - overlap

    for start in range(0, len(words), step_size):

        end = start + chunk_size
        chunk_words = words[start:end]

        if not chunk_words:
            continue

        chunk_text = " ".join(chunk_words).strip()

        if chunk_text:
            chunks.append(chunk_text)

        if end >= len(words):
            break

    return chunks

In [13]:
def create_chunks_with_metadata(
    pages: List[Dict]
) -> List[Dict]:
    """
    Convert extracted pages into chunks with consistent metadata.
    """

    all_chunks = []

    for page in pages:

        document_type = classify_document_type(
            text=page["text"],
            filename=page["filename"]
        )

        page_chunks = split_text_into_chunks(page["text"])

        for chunk_number, chunk_text in enumerate(
            page_chunks,
            start=1
        ):

            chunk_id = (
                f"{page['source_id']}"
                f"_p{page['page_number']}"
                f"_c{chunk_number}"
            )

            all_chunks.append({
                "chunk_id": chunk_id,
                "text": chunk_text,
                "filename": page["filename"],
                "source_id": page["source_id"],
                "document_type": document_type,
                "page_start": page["page_number"],
                "page_end": page["page_number"],
                "chunk_number": chunk_number,
                "extraction_method": page["extraction_method"]
            })

    return all_chunks

In [14]:
QUERY_ROUTING_KEYWORDS = {
    "loan_estimate": [
        "loan estimate",
        "estimated payment",
        "estimated closing cost",
        "projected payment"
    ],

    "closing_disclosure": [
        "closing disclosure",
        "cash to close",
        "final closing costs",
        "amount due at closing"
    ],

    "fees_worksheet": [
        "fees",
        "fee",
        "origination charge",
        "processing fee",
        "underwriting fee",
        "other charges"
    ],

    "mortgage_statement": [
        "amount due",
        "next payment",
        "mortgage balance",
        "escrow balance",
        "late fee"
    ],

    "promissory_note": [
        "interest rate",
        "principal amount",
        "maturity date",
        "monthly payment",
        "promise to pay"
    ],

    "deed_of_trust": [
        "trustee",
        "security instrument",
        "legal description",
        "property used as security"
    ],

    "credit_report": [
        "credit score",
        "credit account",
        "payment history",
        "late payment",
        "credit inquiry"
    ],

    "appraisal": [
        "appraised value",
        "market value",
        "comparable property",
        "property condition"
    ],

    "bank_statement": [
        "bank balance",
        "deposit",
        "withdrawal",
        "ending balance",
        "account balance"
    ],

    "income_document": [
        "income",
        "salary",
        "gross pay",
        "wages",
        "year to date",
        "employer"
    ]
}

In [15]:
def route_query(query: str) -> str:
    """
    Predict which document type should be searched first.
    """

    query_lower = query.lower()
    scores = {}

    for doc_type, keywords in QUERY_ROUTING_KEYWORDS.items():

        score = sum(
            1 for keyword in keywords
            if keyword in query_lower
        )

        scores[doc_type] = score

    best_type = max(scores, key=scores.get)

    if scores[best_type] == 0:
        return "all_documents"

    return best_type

In [16]:
def build_vector_index(
    chunks: List[Dict]
):
    """
    Generate embeddings and store them in a FAISS index.
    """

    if not chunks:
        raise ValueError(
            "No chunks were available for indexing."
        )

    chunk_texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        chunk_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32
    )

    embeddings = embeddings.astype("float32")

    embedding_dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(embedding_dimension)
    index.add(embeddings)

    return index, embeddings

In [17]:
def retrieve_chunks(
    query: str,
    rag_state: Dict,
    top_k: int = 2,
    selected_doc_type: str = "Auto Route",
    minimum_similarity: float = -1.0
) -> Dict:
    """
    Retrieve the most relevant document chunks for a query.

    Improvements:
    - Returns only the top 2 chunks by default
    - Prints retrieval details for debugging
    - Filters out weak matches
    - Keeps the raw cosine similarity score
    """

    if not rag_state:
        return {
            "results": [],
            "route": "none"
        }

    chunks = rag_state.get("chunks", [])
    embeddings = rag_state.get("embeddings")

    if not chunks or embeddings is None:
        return {
            "results": [],
            "route": "none"
        }

    # Determine which document type should be searched
    if selected_doc_type == "Auto Route":
        predicted_route = route_query(query)

    elif selected_doc_type == "All Documents":
        predicted_route = "all_documents"

    else:
        predicted_route = selected_doc_type

    # Select the chunks that match the chosen document type
    if predicted_route == "all_documents":
        eligible_indices = list(range(len(chunks)))

    else:
        eligible_indices = [
            index
            for index, chunk in enumerate(chunks)
            if chunk.get("document_type") == predicted_route
        ]

        # If no chunks match the route, search every document
        if not eligible_indices:
            eligible_indices = list(range(len(chunks)))

            predicted_route = (
                f"{predicted_route} "
                f"(fallback to all documents)"
            )

    # Stop if there are still no eligible chunks
    if not eligible_indices:
        return {
            "results": [],
            "route": predicted_route
        }

    # Get only the embeddings that belong to eligible chunks
    filtered_embeddings = embeddings[eligible_indices].astype(
        "float32"
    )

    # Create an embedding for the user's question
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Create a temporary FAISS index
    # Because embeddings are normalized, inner product acts
    # like cosine similarity
    filtered_index = faiss.IndexFlatIP(
        filtered_embeddings.shape[1]
    )

    filtered_index.add(filtered_embeddings)

    # Never retrieve more chunks than are available
    search_k = min(
        max(top_k, 1),
        len(eligible_indices)
    )

    # Search for the most relevant chunks
    scores, positions = filtered_index.search(
        query_embedding,
        search_k
    )

    # Debugging information
    print("\n" + "=" * 60)
    print(f"QUERY: {query}")
    print(f"ROUTE: {predicted_route}")
    print(f"TOTAL CHUNKS: {len(chunks)}")
    print(f"ELIGIBLE CHUNKS: {len(eligible_indices)}")
    print(f"SEARCH K: {search_k}")
    print(f"MINIMUM SIMILARITY: {minimum_similarity}")
    print(f"FAISS SCORES: {scores[0]}")
    print(f"FAISS POSITIONS: {positions[0]}")
    print("=" * 60)

    results = []

    for rank, (score, position) in enumerate(
        zip(scores[0], positions[0]),
        start=1
    ):
        if position < 0:
            continue

        original_index = eligible_indices[position]
        chunk = chunks[original_index].copy()

        raw_score = float(score)

        chunk["similarity_score"] = raw_score
        chunk["confidence_score"] = max(
            0.0,
            min(1.0, raw_score)
        )

        results.append(chunk)

    return {
        "results": results,
        "route": predicted_route
    }

In [18]:
def build_rag_prompt(
    query: str,
    retrieved_chunks: List[Dict]
) -> str:
    """
    Build a short grounded prompt suitable for FLAN-T5.
    """

    context_sections = []

    # FLAN-T5-base performs better with limited context.
    for number, chunk in enumerate(
        retrieved_chunks[:2],
        start=1
    ):
        # Prevent very large chunks from filling the prompt.
        chunk_text = chunk["text"][:1200]

        context_sections.append(
            f"""
Document {number}
Filename: {chunk["filename"]}
Page: {chunk["page_start"]}
Text: {chunk_text}
""".strip()
        )

    context = "\n\n".join(context_sections)

    prompt = f"""
Use the document text to answer the question.

Give the requested value first.
Then state the filename and page where it was found.

Do not return only a source label.
Do not repeat the question.
Do not invent information.

If the answer is not present, say:
I could not find that information in the uploaded documents.

Documents:
{context}

Question: {query}

Answer:
"""

    return prompt.strip()

In [19]:
def generate_answer(
    prompt: str,
    max_new_tokens: int = 150
) -> str:
    """
    Generate an answer using FLAN-T5.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return answer

In [20]:
import re


def extract_mortgage_field(
    query: str,
    retrieved_chunks: List[Dict]
) -> str | None:
    """
    Extract common mortgage fields directly from retrieved text
    when the language model fails to produce a useful answer.
    """

    combined_text = "\n".join(
        chunk["text"]
        for chunk in retrieved_chunks
    )

    print("\n===== RETRIEVED TEXT =====")
    print(combined_text)
    print("==========================\n")

    query_lower = query.lower()

    # This document displays the loan amount directly before
    # the interest rate, for example:
    # $ 380,000 4.250 %
    amount_rate_match = re.search(
        r"\$\s*([0-9][0-9,]*(?:\.[0-9]{2})?)"
        r"\s+([0-9]+(?:\.[0-9]+)?)\s*%",
        combined_text,
        flags=re.IGNORECASE
    )

    if "loan amount" in query_lower:
        if amount_rate_match:
            loan_amount = amount_rate_match.group(1)
            return f"Loan amount: ${loan_amount}"

        # Backup for normal formats such as:
        # Loan Amount: $380,000
        loan_match = re.search(
            r"(?:total\s+loan\s+amount|loan\s+amount|"
            r"amount\s+financed|principal\s+amount)"
            r"[\s:$-]*\$?\s*"
            r"([0-9][0-9,]*(?:\.[0-9]{2})?)",
            combined_text,
            flags=re.IGNORECASE
        )

        if loan_match:
            return f"Loan amount: ${loan_match.group(1)}"

    elif "interest rate" in query_lower or query_lower.strip() == "rate":
        if amount_rate_match:
            interest_rate = amount_rate_match.group(2)
            return f"Interest rate: {interest_rate}%"

        rate_match = re.search(
            r"(?:interest\s+rate|note\s+rate|annual\s+interest\s+rate)"
            r"[\s:$-]*([0-9]+(?:\.[0-9]+)?)\s*%",
            combined_text,
            flags=re.IGNORECASE
        )

        if rate_match:
            return f"Interest rate: {rate_match.group(1)}%"

    elif "monthly payment" in query_lower or "payment" in query_lower:
        payment_match = re.search(
            r"(?:total\s+estimated\s+monthly\s+payment|"
            r"monthly\s+payment|principal\s+and\s+interest)"
            r".{0,100}?\$?\s*"
            r"([0-9][0-9,]*(?:\.[0-9]{2})?)",
            combined_text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if payment_match:
            return f"Monthly payment: ${payment_match.group(1)}"

    elif "closing cost" in query_lower:
        closing_match = re.search(
            r"(?:estimated\s+closing\s+costs?|"
            r"total\s+closing\s+costs?|closing\s+costs?)"
            r".{0,100}?\$?\s*"
            r"([0-9][0-9,]*(?:\.[0-9]{2})?)",
            combined_text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if closing_match:
            return f"Closing costs: ${closing_match.group(1)}"

    return None

In [21]:
def format_sources(
    retrieved_chunks: List[Dict]
) -> str:
    """
    Format the retrieved sources for display in Gradio.
    """

    if not retrieved_chunks:
        return "No sources retrieved."

    source_lines = []

    seen_sources = set()

    for chunk in retrieved_chunks:

        source_key = (
            chunk["filename"],
            chunk["page_start"],
            chunk["chunk_id"]
        )

        if source_key in seen_sources:
            continue

        seen_sources.add(source_key)

        source_lines.append(
            f"- **{chunk['filename']}** — "
            f"page {chunk['page_start']} | "
            f"type: `{chunk['document_type']}` | "
            f"score: {chunk['confidence_score']:.2%} | "
            f"extraction: {chunk['extraction_method']}"
        )

    return "\n".join(source_lines)

In [22]:
def run_rag_pipeline(
    query: str,
    rag_state: Dict,
    selected_doc_type: str = "Auto Route",
    top_k: int = DEFAULT_TOP_K
) -> Dict:
    """
    Full RAG pipeline:

    1. Retrieve relevant chunks
    2. Build context and prompt
    3. Generate an answer
    4. Use rule-based extraction if needed
    5. Fall back to the best retrieved text
    """

    query = query.strip() if query else ""

    # Check for an empty question
    if not query:
        return {
            "answer": "Please enter a question.",
            "sources": "No sources retrieved.",
            "confidence": 0.0,
            "chunks_used": 0,
            "route": "none"
        }

    # Check that documents have been processed
    if not rag_state or not rag_state.get("chunks"):
        return {
            "answer": (
                "Please upload and process at least one PDF "
                "before asking a question."
            ),
            "sources": "No sources retrieved.",
            "confidence": 0.0,
            "chunks_used": 0,
            "route": "none"
        }

    # Retrieve the best available chunks
    retrieval_output = retrieve_chunks(
        query=query,
        rag_state=rag_state,
        top_k=top_k,
        selected_doc_type=selected_doc_type
    )

    retrieved_chunks = retrieval_output.get("results", [])
    route = retrieval_output.get("route", "none")

    # Stop only if absolutely no text was retrieved
    if not retrieved_chunks:
        return {
            "answer": "I could not retrieve any document text.",
            "sources": "No sources retrieved.",
            "confidence": 0.0,
            "chunks_used": 0,
            "route": route
        }

    # Calculate average retrieval similarity
    confidence_values = [
        chunk.get("confidence_score", 0.0)
        for chunk in retrieved_chunks
    ]

    average_confidence = float(
        np.mean(confidence_values)
    ) if confidence_values else 0.0

    # Build the prompt
    prompt = build_rag_prompt(
        query=query,
        retrieved_chunks=retrieved_chunks
    )

    # Generate an answer using the language model
    try:
        answer = generate_answer(prompt)
    except Exception as error:
        print(f"Generation error: {error}")
        answer = ""

    answer = answer.strip() if answer else ""
    answer_lower = answer.lower()

    # Detect weak or unusable model responses
    source_only = (
        not answer
        or answer_lower.startswith("[source")
        or answer_lower.startswith("source ")
        or answer_lower.startswith("[document")
        or answer_lower.startswith("document ")
        or "filename:" in answer_lower
        or "could not identify a clear answer" in answer_lower
        or "could not find sufficiently relevant information" in answer_lower
        or len(answer.split()) < 5
    )

    # Try rule-based extraction when the model answer is weak
    if source_only:
        extracted_answer = extract_mortgage_field(
            query=query,
            retrieved_chunks=retrieved_chunks
        )

        if extracted_answer:
            best_source = retrieved_chunks[0]

            filename = best_source.get(
                "filename",
                "the retrieved document"
            )

            page = best_source.get(
                "page_start",
                best_source.get("page_number", "Unknown")
            )

            answer = (
                f"{extracted_answer}\n\n"
                f"Found in {filename}, page {page}."
            )

        else:
            # Final fallback: return the best retrieved text
            best_chunk = retrieved_chunks[0]
            best_text = best_chunk.get("text", "").strip()

            if best_text:
                answer = (
                    "Here is the most relevant document section I found:\n\n"
                    + best_text[:1500]
                )
            else:
                answer = (
                    "I retrieved a document section, but it did not "
                    "contain readable text."
                )

    # Final safety check
    if not answer:
        best_text = retrieved_chunks[0].get("text", "").strip()

        answer = (
            "Here is the most relevant document section I found:\n\n"
            + best_text[:1500]
        )

    return {
        "answer": answer,
        "sources": format_sources(retrieved_chunks),
        "confidence": average_confidence,
        "chunks_used": len(retrieved_chunks),
        "route": route
    }

In [23]:
def process_uploaded_documents(
    files,
    progress=gr.Progress()
):
    """
    Extract, OCR, classify, chunk, embed, and index uploaded PDFs.
    """

    empty_state = {
        "chunks": [],
        "embeddings": None,
        "filenames": [],
        "document_types": []
    }

    if not files:
        return (
            "❌ Please upload at least one PDF.",
            empty_state,
            "No document summary available."
        )

    if isinstance(files, str):
        files = [files]

    all_pages = []
    processed_files = []
    failed_files = []

    for file_number, file_path in enumerate(files, start=1):

        filename = os.path.basename(file_path)

        progress(
            (file_number - 1) / len(files),
            desc=(
                f"Starting document "
                f"{file_number}/{len(files)}: {filename}"
            )
        )

        try:
            pages = extract_pdf_pages(
                pdf_path=file_path,
                progress=progress
            )

            if pages:
                all_pages.extend(pages)
                processed_files.append(filename)
            else:
                failed_files.append(
                    f"{filename}: no readable text found"
                )

        except Exception as error:
            failed_files.append(
                f"{filename}: {str(error)}"
            )

    if not all_pages:

        return (
            "❌ No readable text was extracted.",
            empty_state,
            "\n".join(failed_files)
        )

    progress(
        0.75,
        desc="Cleaning and chunking extracted text"
    )

    chunks = create_chunks_with_metadata(all_pages)

    if not chunks:

        return (
            "❌ Text was extracted, but no chunks were created.",
            empty_state,
            "Check the uploaded documents."
        )

    progress(
        0.85,
        desc="Generating embeddings and building FAISS index"
    )

    vector_index, embeddings = build_vector_index(chunks)

    document_types = sorted(
        set(chunk["document_type"] for chunk in chunks)
    )

    digital_pages = sum(
        page["extraction_method"] == "digital"
        for page in all_pages
    )

    ocr_pages = sum(
        page["extraction_method"] == "ocr"
        for page in all_pages
    )

    rag_state = {
        "chunks": chunks,
        "embeddings": embeddings,
        "filenames": processed_files,
        "document_types": document_types
    }

    status = (
        f"✅ **Processing complete**\n\n"
        f"- PDFs processed: **{len(processed_files)}**\n"
        f"- Pages extracted: **{len(all_pages)}**\n"
        f"- Digital pages: **{digital_pages}**\n"
        f"- OCR pages: **{ocr_pages}**\n"
        f"- Searchable chunks: **{len(chunks)}**\n"
        f"- Document types found: **{len(document_types)}**"
    )

    if failed_files:
        status += "\n\n⚠️ **Files with issues:**\n"
        status += "\n".join(
            f"- {item}" for item in failed_files
        )

    summary = (
        "### Indexed document summary\n\n"
        f"**Files:** {', '.join(processed_files)}\n\n"
        f"**Document types:** "
        f"{', '.join(document_types)}\n\n"
        f"**Embedding model:** `{EMBEDDING_MODEL_NAME}`\n\n"
        f"**Language model:** `{LLM_MODEL_NAME}`"
    )

    progress(
        1.0,
        desc="Documents are ready"
    )

    return status, rag_state, summary

In [24]:
def chat_with_documents(
    message,
    history,
    rag_state,
    selected_doc_type,
    top_k
):
    """
    Handle a user message and update Gradio chat history.
    """

    if history is None:
        history = []

    message = message.strip() if message else ""

    if not message:
        return (
            history,
            "",
            "No answer metrics available.",
            "No sources retrieved."
        )

    updated_history = history + [
        {
            "role": "user",
            "content": message
        }
    ]

    try:
        result = run_rag_pipeline(
            query=message,
            rag_state=rag_state,
            selected_doc_type=selected_doc_type,
            top_k=int(top_k)
        )

        answer = result["answer"]

        updated_history.append({
            "role": "assistant",
            "content": answer
        })

        metrics = (
            f"### Retrieval details\n\n"
            f"- **Confidence:** "
            f"{result['confidence']:.2%}\n"
            f"- **Chunks used:** "
            f"{result['chunks_used']}\n"
            f"- **Route:** "
            f"`{result['route']}`\n"
            f"- **Selected filter:** "
            f"`{selected_doc_type}`"
        )

        sources = (
            "### Supporting sources\n\n"
            + result["sources"]
        )

        return (
            updated_history,
            "",
            metrics,
            sources
        )

    except Exception as error:

        error_answer = (
            "An error occurred while answering the question: "
            f"{str(error)}"
        )

        updated_history.append({
            "role": "assistant",
            "content": error_answer
        })

        return (
            updated_history,
            "",
            "The pipeline encountered an error.",
            "No sources available."
        )

In [25]:
def clear_chat_history():
    """
    Clear the conversation and answer information.
    """

    return (
        [],
        "",
        "No answer metrics available.",
        "No sources retrieved.",
        None
    )

In [26]:
def save_chat_history(history):
    """
    Save the current conversation as a text file.
    """

    if not history:
        return None

    timestamp = datetime.now().strftime(
        "%Y-%m-%d_%H-%M-%S"
    )

    file_path = os.path.join(
        tempfile.gettempdir(),
        f"mortgage_rag_chat_{timestamp}.txt"
    )

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write("MORTGAGE DOCUMENT RAG CHAT\n")
        file.write("=" * 60 + "\n\n")

        for item in history:

            role = item.get(
                "role",
                "unknown"
            ).upper()

            content = item.get(
                "content",
                ""
            )

            file.write(f"{role}\n")
            file.write("-" * 20 + "\n")
            file.write(f"{content}\n\n")

    return file_path

In [27]:
DOCUMENT_FILTER_CHOICES = [
    "Auto Route",
    "All Documents",
    "loan_estimate",
    "closing_disclosure",
    "fees_worksheet",
    "mortgage_statement",
    "promissory_note",
    "deed_of_trust",
    "credit_report",
    "appraisal",
    "bank_statement",
    "income_document",
    "other_mortgage_document"
]

In [28]:
custom_css = """
.gradio-container {
    max-width: 1350px !important;
    margin: 0 auto !important;
}

#main-title {
    text-align: center;
    padding: 22px;
    border-radius: 18px;
    background: linear-gradient(
        90deg,
        #16325c,
        #2458a6
    );
    color: white;
    margin-bottom: 14px;
}

#subtitle-box {
    text-align: center;
    padding: 10px;
    border-radius: 12px;
}

#process-button,
#ask-button {
    font-weight: 700;
    border-radius: 10px;
}

.section-card {
    border-radius: 14px;
}
"""

In [29]:
empty_rag_state = {
    "chunks": [],
    "embeddings": None,
    "filenames": [],
    "document_types": []
}

In [30]:
with gr.Blocks(
    title="Mortgage Document RAG Assistant"
) as demo:

    rag_state = gr.State(empty_rag_state)

    gr.Markdown(
        """
        # 🏠 Mortgage Document Intelligence Assistant

        Upload digital or scanned mortgage documents and ask grounded
        questions about rates, payments, fees, balances, income,
        closing costs, and other document details.
        """,
        elem_id="main-title"
    )

    gr.Markdown(
        """
        This assistant uses OCR, metadata routing, semantic retrieval,
        FAISS, and an open-source language model. Answers should be
        verified against the displayed document sources.
        """,
        elem_id="subtitle-box"
    )

    with gr.Row():

        # ---------------------------------------------------------
        # LEFT COLUMN: DOCUMENT PROCESSING
        # ---------------------------------------------------------
        with gr.Column(
            scale=1,
            min_width=330
        ):

            gr.Markdown("## 📄 Document processing")

            pdf_files = gr.File(
                label="Upload mortgage PDFs",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath"
            )

            process_button = gr.Button(
                "⚙️ Process and Index Documents",
                variant="primary",
                elem_id="process-button"
            )

            process_status = gr.Markdown(
                "No documents have been processed."
            )

            document_summary = gr.Markdown(
                "No document summary available."
            )

            gr.Markdown("## 🎯 Retrieval controls")

            document_filter = gr.Dropdown(
                choices=DOCUMENT_FILTER_CHOICES,
                value="Auto Route",
                label="Document-type routing",
                info=(
                    "Auto Route predicts the most relevant "
                    "mortgage-document type."
                )
            )

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=8,
                value=2,
                step=1,
                label="Number of chunks to retrieve"
            )

        # ---------------------------------------------------------
        # RIGHT COLUMN: CHATBOT
        # ---------------------------------------------------------
        with gr.Column(
            scale=2,
            min_width=600
        ):

            gr.Markdown("## 💬 Ask your documents")

            chatbot = gr.Chatbot(
                label="Mortgage document conversation",
                height=480,
                placeholder=(
                    "Process your PDFs, then ask a question such as: "
                    "'What is the loan amount?'"
                )
            )

            user_question = gr.Textbox(
                label="Question",
                placeholder=(
                    "Example: What is the interest rate "
                    "and which document contains it?"
                ),
                lines=2
            )

            with gr.Row():

                ask_button = gr.Button(
                    "🔎 Ask Documents",
                    variant="primary",
                    elem_id="ask-button"
                )

                clear_button = gr.Button(
                    "🗑️ Clear Conversation"
                )

            with gr.Accordion(
                "📊 Retrieval confidence and routing",
                open=True
            ):

                answer_metrics = gr.Markdown(
                    "No answer metrics available."
                )

            with gr.Accordion(
                "📚 Sources used for the answer",
                open=True
            ):

                answer_sources = gr.Markdown(
                    "No sources retrieved."
                )

            with gr.Row():

                save_button = gr.Button(
                    "💾 Prepare Chat History"
                )

                download_button = gr.DownloadButton(
                    label="⬇️ Download Chat History",
                    value=None
                )

    # -------------------------------------------------------------
    # EVENT: PROCESS DOCUMENTS
    # -------------------------------------------------------------
    process_button.click(
        fn=process_uploaded_documents,
        inputs=pdf_files,
        outputs=[
            process_status,
            rag_state,
            document_summary
        ],
        show_progress="full"
    )

    # -------------------------------------------------------------
    # EVENT: ASK WITH BUTTON
    # -------------------------------------------------------------
    ask_button.click(
        fn=chat_with_documents,
        inputs=[
            user_question,
            chatbot,
            rag_state,
            document_filter,
            top_k_slider
        ],
        outputs=[
            chatbot,
            user_question,
            answer_metrics,
            answer_sources
        ],
        show_progress="full"
    )

    # -------------------------------------------------------------
    # EVENT: ASK BY PRESSING ENTER
    # -------------------------------------------------------------
    user_question.submit(
        fn=chat_with_documents,
        inputs=[
            user_question,
            chatbot,
            rag_state,
            document_filter,
            top_k_slider
        ],
        outputs=[
            chatbot,
            user_question,
            answer_metrics,
            answer_sources
        ],
        show_progress="full"
    )

    # -------------------------------------------------------------
    # EVENT: CLEAR
    # -------------------------------------------------------------
    clear_button.click(
        fn=clear_chat_history,
        inputs=None,
        outputs=[
            chatbot,
            user_question,
            answer_metrics,
            answer_sources,
            download_button
        ]
    )

    # EVENT: SAVE CHAT

    save_button.click(
        fn=save_chat_history,
        inputs=chatbot,
        outputs=download_button
    )

In [ ]:
demo.queue().launch(
    share=True,
    debug=True,
    css=custom_css
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c97d08cf0de3a5944.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


QUERY: what is the loan amount
ROUTE: all_documents
TOTAL CHUNKS: 6
ELIGIBLE CHUNKS: 6
SEARCH K: 2
MINIMUM SIMILARITY: -1.0
FAISS SCORES: [0.4964609  0.47057402]
FAISS POSITIONS: [3 4]

===== RETRIEVED TEXT =====
Your actual rate, payment, and cost could be higher. Get an official Loan Estimate before choosing a loan. Fee Details and Summary Applicants: Application No: Date Prepared: Loan Program: Prepared By: THIS IS NOT A GOOD FAITH ESTIMATE (GFE). This "Fees Worksheet" is provided for informational purposes ONLY, to assist you in determining an estimate of cash that may be required to close and an estimate of your proposed monthly mortgage payment. Actual charges may be more or less, and your transaction may not involve a fee for every item listed. Total Loan Amount: Interest Rate: Term/Due In: Fee Paid To Paid By (Fee Split**) Amount PFC / F / POC TOTAL ESTIMATED FUNDS NEEDED TO CLOSE: TOTAL ESTIMATED MONTHLY PAYMENT: Total Estimated Funds Total Monthly Payment Purchase Price (+) 